# PlantSeg Round-2 — Notebook H: Stage-2 segmenter checkpoint verification (inference only, all 6 models)

**What this is.** Unlike `02_Stage2_Segmentation/`'s training notebooks, this notebook does
**no training**. It loads the 6 currently-deposited Stage-2 checkpoints and re-evaluates each
one on the held-out test set with the exact same metric code the original training notebooks
used, then compares the result against two independent references: the preserved original
`s2_result_{Model}.pkl` record for each model, and the paper's Table (`tab:seg`).

**Why this exists.** The classifier side of this project (Notebooks F/G) found that 6 of 8
classifier checkpoints did not reproduce their originally-reported figure, because the
checkpoint file available for re-evaluation was not always the exact weights behind the
original number (a checkpoint-deposit/bookkeeping issue, not a correctness problem — the
original per-image record proved every figure was genuine regardless). The paper's own
Limitations text already discloses a related, smaller-scale version of this for segmentation:
4 of 6 segmenter checkpoints were independently confirmed to reproduce Table tab:seg to within
±0.0001 during the cross-link analysis, but the other 2 (DeepLabV3+/EfficientNet-B3 and
U-Net/ResNet34) had their original checkpoints unavailable at that time and had to be
retrained from scratch, landing close but not identical to the original figure (0.630 vs 0.648,
and 0.613 vs 0.602). This notebook checks where the *currently deposited* checkpoints for all
6 models actually stand today, as one short, genuinely-executed inference run — not a
retrain (training these models originally took 1.8–9.2 hours each; this notebook should
take well under an hour total since it is forward-pass-only).

**If you're re-running after a stall with zero log output:** this notebook prints a heartbeat
at every step (`flush=True` everywhere) and skips `pip install` if the package is already
importable, so whatever happens next, the log shows exactly where it stopped.

**Inputs needed (attach as Kaggle datasets before running):**
1. The main `plantseg` dataset (same one every other notebook in this project uses — needs
   `Metadata.csv`, `annotation_{train,val,test}.json`, and the `images/` folder).
2. A dataset containing the 6 Stage-2 checkpoint files: `UNet_resnet34.pt`,
   `UNetPP_resnet50.pt`, `FPN_resnet50.pt`, `DeepLabV3Plus_resnet50.pt`,
   `DeepLabV3Plus_efficientnet-b3.pt`, `SegFormer_b2.pt` (source:
   `Old notebooks and results/Stage 2/checkpoints/` on the local project folder — these aren't
   on Kaggle yet, so create a new private Kaggle Dataset from them and attach it).

## Setup

In [1]:
import importlib, subprocess, sys, time

def heartbeat(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

heartbeat("Notebook H starting: Stage-2 checkpoint verification (inference only)")

for pkg, import_name in [("segmentation-models-pytorch", "segmentation_models_pytorch"),
                          ("albumentations", "albumentations")]:
    try:
        importlib.import_module(import_name)
        heartbeat(f"{import_name} already importable, skipping pip install")
    except ImportError:
        heartbeat(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        heartbeat(f"{pkg} installed")

heartbeat("imports starting")
import os, json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
heartbeat(f"imports done, device={device}")

[14:03:27] Notebook H starting: Stage-2 checkpoint verification (inference only)
[14:03:27] installing segmentation-models-pytorch ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
[14:03:32] segmentation-models-pytorch installed
[14:03:42] albumentations already importable, skipping pip install
[14:03:42] imports starting
[14:04:05] imports done, device=cuda


## Config and input auto-discovery

In [2]:
# ============ CONFIG ============
SEG_TASK = "binary"; NUM_SEG = 2; IMG_SIZE = 512
OUTDIR = Path("/kaggle/working/s2_verify"); MASK = OUTDIR/"masks"
for d in [OUTDIR, MASK]: d.mkdir(parents=True, exist_ok=True)

# Auto-discover the main plantseg dataset (same pattern as earlier round-2 notebooks)
DATA_ROOT = None
for c in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"), Path("/kaggle/input/plantseg"),
          Path("plantseg"), Path(".")]:
    if (c/"Metadata.csv").exists():
        DATA_ROOT = c; break
assert DATA_ROOT is not None, "Metadata.csv not found under any expected path — attach the plantseg dataset."
heartbeat(f"DATA_ROOT = {DATA_ROOT}")

# Auto-discover the Stage-2 checkpoint dataset: search /kaggle/input for the 6 expected filenames
CKPT_NAMES = ["UNet_resnet34", "UNetPP_resnet50", "FPN_resnet50",
              "DeepLabV3Plus_resnet50", "DeepLabV3Plus_efficientnet-b3", "SegFormer_b2"]
CKPT_PATHS = {}
search_roots = [Path("/kaggle/input"), Path(".")]
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*.pt"):
        stem = p.stem
        if stem in CKPT_NAMES and stem not in CKPT_PATHS:
            CKPT_PATHS[stem] = p
missing = [n for n in CKPT_NAMES if n not in CKPT_PATHS]
assert not missing, f"Missing checkpoint files for: {missing} — attach the Stage-2 checkpoints dataset."
for n in CKPT_NAMES:
    heartbeat(f"checkpoint found: {n} -> {CKPT_PATHS[n]}")

[14:04:05] DATA_ROOT = /kaggle/input/datasets/alenmaruf/plantseg/plantseg
[14:04:40] checkpoint found: UNet_resnet34 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/UNet_resnet34.pt
[14:04:40] checkpoint found: UNetPP_resnet50 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/UNetPP_resnet50.pt
[14:04:40] checkpoint found: FPN_resnet50 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/FPN_resnet50.pt
[14:04:40] checkpoint found: DeepLabV3Plus_resnet50 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/DeepLabV3Plus_resnet50.pt
[14:04:40] checkpoint found: DeepLabV3Plus_efficientnet-b3 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/DeepLabV3Plus_efficientnet-b3.pt
[14:04:40] checkpoint found: SegFormer_b2 -> /kaggle/input/datasets/alenmaruf/stage2-checkpoints/SegFormer_b2.pt


## Index + binary masks (identical to the Stage-2 training notebooks' code)

In [3]:
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training": "train", "Validation": "val", "Test": "test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)

def load_anns(split):
    d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
    for a in d["annotations"]:
        by[a["image_id"]].append(a)
    fn = {}
    for im in d["images"]:
        polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
        fn[im["file_name"]] = (im["width"], im["height"], polys)
    return fn

def rasterize(w, h, polys):
    m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
    for seg in polys:
        dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
    return m

heartbeat("building test-split binary masks ...")
fn = load_anns("test"); od = MASK/"test"; od.mkdir(parents=True, exist_ok=True)
recs = []
for r in meta[meta.split == "test"].itertuples():
    w, h, polys = fn.get(r.Name, (None, None, [])); mp = od/(Path(r.Name).stem + ".png")
    if not mp.exists():
        if w is None:
            with Image.open(r.filepath) as i0:
                w, h = i0.size
        rasterize(w, h, polys).save(mp)
    recs.append({"name": r.Name, "filepath": str(r.filepath), "mask": str(mp)})
test_df = pd.DataFrame(recs)
heartbeat(f"test set ready: {len(test_df)} images (expect 1561)")

[14:04:41] building test-split binary masks ...
[14:04:58] test set ready: 1561 images (expect 1561)


## Dataset / loader (test split only — this notebook never touches train/val)

In [4]:
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()])

class SegDS(Dataset):
    def __init__(s, d, t): s.df = d.reset_index(drop=True); s.t = t
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        img = np.array(Image.open(r.filepath).convert("RGB")); m = np.array(Image.open(r["mask"]))
        a = s.t(image=img, mask=m)
        return a["image"], a["mask"].long()

def test_loader(bs):
    return DataLoader(SegDS(test_df, eval_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

heartbeat("test loader ready")

[14:04:58] test loader ready


## Model (identical SegNet wrapper the checkpoints were saved from)

In [5]:
SMP_SPECS = {"UNet_resnet34": (smp.Unet, "resnet34"), "UNetPP_resnet50": (smp.UnetPlusPlus, "resnet50"),
    "DeepLabV3Plus_resnet50": (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50": (smp.FPN, "resnet50")}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}

class SegNet(nn.Module):
    def __init__(s, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            s.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); s.kind = "smp"
        else:
            s.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); s.kind = "hf"
    def forward(s, x):
        if s.kind == "smp":
            return s.net(x)
        lg = s.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

def bs_for(name): return 6 if ("b2" in name or "b4" in name) else 8

heartbeat("model builder ready (encoder_weights=None: we load trained weights next, no imagenet init needed)")

[14:04:58] model builder ready (encoder_weights=None: we load trained weights next, no imagenet init needed)


## Metrics (identical Conf/evaluate code the training notebooks used, so numbers are directly comparable)

In [6]:
class Conf:
    def __init__(s, n): s.n = n; s.cm = np.zeros((n, n), np.int64)
    def add(s, gt, pr):
        k = (gt >= 0) & (gt < s.n)
        s.cm += np.bincount(s.n * gt[k].astype(int) + pr[k], minlength=s.n ** 2).reshape(s.n, s.n)
    def scores(s):
        cm = s.cm.astype(float); tp = np.diag(cm)
        iou = tp / (cm.sum(1) + cm.sum(0) - tp + 1e-9); dice = 2 * tp / (cm.sum(1) + cm.sum(0) + 1e-9)
        pr = cm.sum(1) > 0
        return {"mIoU": float(np.nanmean(iou[pr])), "mDice": float(np.nanmean(dice[pr])),
                "pixel_acc": float(tp.sum() / (cm.sum() + 1e-9)),
                "fg_IoU": float(iou[1]), "fg_Dice": float(dice[1])}, iou

def evaluate(model, loader, n):
    model.eval(); cf = Conf(n)
    with torch.no_grad():
        for x, y in loader:
            pr = model(x.to(device)).argmax(1).cpu().numpy()
            cf.add(y.numpy().ravel(), pr.ravel())
    return cf

heartbeat("metrics ready")

[14:04:58] metrics ready


## Reference values (paper Table tab:seg, and the preserved original Stage-2 result pkls)

In [7]:
# paper Table tab:seg, columns: fg-IoU, fg-Dice, mIoU, Pixel Acc.
PAPER_TABLE = {
    "DeepLabV3Plus_efficientnet-b3":   {"fg_IoU": 0.648, "fg_Dice": 0.787, "mIoU": 0.770, "pixel_acc": 0.910},
    "FPN_resnet50":                    {"fg_IoU": 0.614, "fg_Dice": 0.760, "mIoU": 0.747, "pixel_acc": 0.900},
    "UNet_resnet34":                   {"fg_IoU": 0.602, "fg_Dice": 0.751, "mIoU": 0.741, "pixel_acc": 0.898},
    "UNetPP_resnet50":                 {"fg_IoU": 0.595, "fg_Dice": 0.746, "mIoU": 0.737, "pixel_acc": 0.897},
    "DeepLabV3Plus_resnet50":          {"fg_IoU": 0.590, "fg_Dice": 0.742, "mIoU": 0.730, "pixel_acc": 0.891},
    "SegFormer_b2":                    {"fg_IoU": 0.579, "fg_Dice": 0.734, "mIoU": 0.723, "pixel_acc": 0.887},
}
heartbeat("reference table loaded (verbatim from paper.tex Table tab:seg)")

[14:04:58] reference table loaded (verbatim from paper.tex Table tab:seg)


## Main loop: load each checkpoint, evaluate on test set, compare

In [8]:
results = {}
for name in CKPT_NAMES:
    heartbeat(f"--- {name}: loading checkpoint ---")
    model = SegNet(name, NUM_SEG).to(device)
    state = torch.load(CKPT_PATHS[name], map_location=device)
    missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
    if missing_keys or unexpected_keys:
        heartbeat(f"  WARNING non-strict load: missing={missing_keys[:3]}... unexpected={unexpected_keys[:3]}...")
    else:
        heartbeat("  state_dict loaded strictly, 0 missing/unexpected keys")

    heartbeat(f"--- {name}: running test-set inference ({len(test_df)} images) ---")
    t0 = time.time()
    cf = evaluate(model, test_loader(bs_for(name)), NUM_SEG)
    sc, iou = cf.scores()
    elapsed = time.time() - t0
    heartbeat(f"--- {name}: done in {elapsed:.1f}s | fresh test_scores = {{k: round(v,4) for k,v in sc.items()}} ---")
    results[name] = {"fresh_test_scores": sc, "elapsed_sec": elapsed}

heartbeat("all 6 models evaluated")

[14:04:58] --- UNet_resnet34: loading checkpoint ---
[14:04:59]   state_dict loaded strictly, 0 missing/unexpected keys
[14:04:59] --- UNet_resnet34: running test-set inference (1561 images) ---
[14:05:34] --- UNet_resnet34: done in 35.0s | fresh test_scores = {k: round(v,4) for k,v in sc.items()} ---
[14:05:34] --- UNetPP_resnet50: loading checkpoint ---
[14:05:37]   state_dict loaded strictly, 0 missing/unexpected keys
[14:05:37] --- UNetPP_resnet50: running test-set inference (1561 images) ---
[14:08:13] --- UNetPP_resnet50: done in 156.1s | fresh test_scores = {k: round(v,4) for k,v in sc.items()} ---
[14:08:13] --- FPN_resnet50: loading checkpoint ---
[14:08:14]   state_dict loaded strictly, 0 missing/unexpected keys
[14:08:14] --- FPN_resnet50: running test-set inference (1561 images) ---
[14:08:53] --- FPN_resnet50: done in 38.8s | fresh test_scores = {k: round(v,4) for k,v in sc.items()} ---
[14:08:53] --- DeepLabV3Plus_resnet50: loading checkpoint ---
[14:08:54]   state_dict l

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 99.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.weight                           

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

[14:10:27]   WARNING non-strict load: missing=['net.segformer.stages.0.patch_embeddings.proj.weight', 'net.segformer.stages.0.patch_embeddings.proj.bias', 'net.segformer.stages.0.patch_embeddings.layer_norm.weight']... unexpected=['net.segformer.encoder.patch_embeddings.0.proj.weight', 'net.segformer.encoder.patch_embeddings.0.proj.bias', 'net.segformer.encoder.patch_embeddings.0.layer_norm.weight']...
[14:10:27] --- SegFormer_b2: running test-set inference (1561 images) ---
[14:11:54] --- SegFormer_b2: done in 87.1s | fresh test_scores = {k: round(v,4) for k,v in sc.items()} ---
[14:11:54] all 6 models evaluated


## Compare against the preserved original pkl (attach the `result pkl` folder as a third Kaggle input, or fill in manually)

In [9]:
# Original per-model test_scores, from Old notebooks and results/Stage 2/result pkl/s2_result_{Model}.pkl
# (copied in here directly so this notebook doesn't need a third dataset attached; verified 2026-10-10)
ORIGINAL_PKL_SCORES = {
    "UNet_resnet34":                 {"mIoU": 0.7408, "fg_IoU": 0.6015, "mDice": 0.8437, "pixel_acc": 0.8984},
    "DeepLabV3Plus_resnet50":        {"mIoU": 0.7303, "fg_IoU": 0.5904, "mDice": 0.8365, "pixel_acc": 0.8906},
    "FPN_resnet50":                  {"mIoU": 0.7474, "fg_IoU": 0.6135, "mDice": 0.8487, "pixel_acc": 0.9002},
    "SegFormer_b2":                  {"mIoU": 0.7226, "fg_IoU": 0.5792, "mDice": 0.8309, "pixel_acc": 0.8869},
    "UNetPP_resnet50":               {"mIoU": 0.7369, "fg_IoU": 0.5954, "mDice": 0.8408, "pixel_acc": 0.8968},
    "DeepLabV3Plus_efficientnet-b3": {"mIoU": 0.7704, "fg_IoU": 0.6484, "mDice": 0.8649, "pixel_acc": 0.9102},
}

print(f"{'Model':32s} {'fresh fg-IoU':>13s} {'orig pkl fg-IoU':>16s} {'paper fg-IoU':>13s} {'status':>10s}")
for name in CKPT_NAMES:
    fresh = results[name]["fresh_test_scores"]["fg_IoU"]
    orig = ORIGINAL_PKL_SCORES[name]["fg_IoU"]
    paper = PAPER_TABLE[name]["fg_IoU"]
    diff_orig = abs(fresh - orig)
    status = "OK" if diff_orig < 0.01 else ("gap" if diff_orig < 0.05 else "LARGE GAP")
    paper_str = f"{paper:.3f}"
    print(f"{name:32s} {fresh:13.4f} {orig:16.4f} {paper_str:>13s} {status:>10s}")

heartbeat("comparison table printed above")

Model                             fresh fg-IoU  orig pkl fg-IoU  paper fg-IoU     status
UNet_resnet34                           0.6129           0.6015         0.602        gap
UNetPP_resnet50                         0.5954           0.5954         0.595         OK
FPN_resnet50                            0.6135           0.6135         0.614         OK
DeepLabV3Plus_resnet50                  0.5904           0.5904         0.590         OK
DeepLabV3Plus_efficientnet-b3           0.6304           0.6484         0.648        gap
SegFormer_b2                            0.2100           0.5792         0.579  LARGE GAP
[14:11:54] comparison table printed above


## Save verification record

In [10]:
out = {"checkpoint_paths": {k: str(v) for k, v in CKPT_PATHS.items()},
       "results": results, "original_pkl_scores": ORIGINAL_PKL_SCORES, "paper_table": PAPER_TABLE,
       "run_timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
with open(OUTDIR/"notebookH_stage2_checkpoint_verification.pkl", "wb") as f:
    pickle.dump(out, f)
heartbeat(f"SAVED {OUTDIR/'notebookH_stage2_checkpoint_verification.pkl'}")
heartbeat("Notebook H complete. Bring this pkl back to the project folder for the paper/README update.")

[14:11:54] SAVED /kaggle/working/s2_verify/notebookH_stage2_checkpoint_verification.pkl
[14:11:54] Notebook H complete. Bring this pkl back to the project folder for the paper/README update.
